In [1]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, metrics

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.21.0


In [6]:
PROCESSED_DATA_PATH = "../processed_data"
ds_train = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'train'))
ds_val = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'val'))
ds_test = tf.data.Dataset.load(os.path.join(PROCESSED_DATA_PATH, 'test'))

In [5]:
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

NUM_CLASSES = 5

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', metrics.SparseTopKCategoricalAccuracy(k=1, name='mAP')]
)

In [7]:
EPOCHS = 50
print(f"🚀 Launching DenseNet121 training pipeline...")
start_time = time.time()

history = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS)

end_time = time.time()
densenet_total_time = end_time - start_time
print(f"\nTotal DenseNet121 Training Time: {densenet_total_time:.2f} seconds")

🚀 Launching DenseNet121 training pipeline...
Epoch 1/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.7690 - loss: 0.6531 - mAP: 0.7690 - val_accuracy: 0.9522 - val_loss: 0.2504 - val_mAP: 0.9522
Epoch 2/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 450s 3s/step - accuracy: 0.9468 - loss: 0.2122 - mAP: 0.9468 - val_accuracy: 0.9779 - val_loss: 0.1385 - val_mAP: 0.9779
Epoch 3/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 447s 3s/step - accuracy: 0.9623 - loss: 0.1436 - mAP: 0.9623 - val_accuracy: 0.9835 - val_loss: 0.0994 - val_mAP: 0.9835
Epoch 4/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 644s 4s/step - accuracy: 0.9726 - loss: 0.1073 - mAP: 0.9726 - val_accuracy: 0.9871 - val_loss: 0.0786 - val_mAP: 0.9871
Epoch 5/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 534s 4s/step - accuracy: 0.9775 - loss: 0.0920 - mAP: 0.9775 - val_accuracy: 0.9890 - val_loss: 0.0661 - val_mAP: 0.9890
Epoch 6/50
134/134 ━━━━━━━━━━━━━━━━━━━━ 546s 4s/step - accuracy: 0.9810 - loss: 0.0745 - mAP: 0.9810 - val_accuracy: 0.9908 - val_loss: 0.0568 - val_m

In [8]:
import os
import csv

# Ensure the results directory exists
os.makedirs('../results', exist_ok=True)

# 1. Capture the history dictionary from your DenseNet run
history_dict = history.history
csv_path = '../results/densenet121_history.csv'

# 2. Extract metrics and write them row-by-row to CSV
with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    keys = list(history_dict.keys())
    writer.writerow(keys) # Header row
    
    num_epochs = len(history_dict[keys[0]])
    for i in range(num_epochs):
        row = [history_dict[k][i] for k in keys]
        writer.writerow(row)

# 3. Save the model configuration
model.save('../results/densenet121_model.keras')

# 4. Log the execution time
with open('../results/densenet121_time.txt', 'w') as f:
    f.write(str(densenet_total_time))

print("💾 SUCCESS! DenseNet121 metrics exported to CSV and model architecture saved.")

💾 SUCCESS! DenseNet121 metrics exported to CSV and model architecture saved.
